#Import Libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold

#Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
OUTPUT_DIR = Path("/content/drive/MyDrive/Simple")

#Vars

In [4]:
SEED = 55
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

#Get Dataset file

In [5]:
df = pd.read_excel("/content/drive/MyDrive/Simple/All_data.xlsx")
print("Shape:", df.shape)
display(df.head(10))

Shape: (104531, 5)


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Sentence #,word,aspect,categort,polarity
0,Sentence:0,عمال,O,O,O
1,Sentence:0,شركات,O,O,O
2,Sentence:0,التوصيل,O,O,O
3,Sentence:0,أمثال,O,O,O
4,Sentence:0,ذبحونا,B,Service,Neg
5,Sentence:0,ابعث,O,O,O
6,Sentence:0,لي,O,O,O
7,Sentence:0,لوكيشن,B,Location,Neg
8,Sentence:0,في,O,O,O
9,Sentence:0,كل,O,O,O


#Schema and order

In [6]:
df.columns = df.columns.str.strip()

raw = pd.DataFrame({
    "sentence_id": df["Sentence #"],
    "word": df["word"],
    "bio_tag": df["aspect"],
    "category": df["categort"],
    "polarity": df["polarity"],
})

#CLEAN + Count

In [7]:
raw["source_row"] = np.arange(len(raw))
raw = raw.dropna(subset=["sentence_id", "word"]).copy()
raw["sentence_id"] = raw["sentence_id"].astype(str).str.strip()
raw["word"] = raw["word"].astype(str).str.strip()
raw["bio_tag"] = raw["bio_tag"].astype(str).str.strip()
raw["category"] = raw["category"].astype(str).str.strip()
raw["polarity"] = raw["polarity"].astype(str).str.strip()
raw = raw[raw["word"] != ""].copy()

print("Rows:", len(raw))
print("Unique sentences:", raw["sentence_id"].nunique())
print("\nBIO tag values:")
display(raw["bio_tag"].value_counts().reset_index())
print("\nPolarity values:")
display(raw["polarity"].value_counts().reset_index())
print("\nCategory values:")
display(raw["category"].value_counts().reset_index())

Rows: 104531
Unique sentences: 4846

BIO tag values:


,bio_tag,count
0,O,80798
1,I,13649
2,B,10084



Polarity values:


,polarity,count
0,O,80798
1,Neg,17965
2,Pos,5768



Category values:


,category,count
0,O,80799
1,Service,12071
2,Price,5102
3,Time,5091
4,Location,1468


In [8]:
import plotly.express as px

fig = px.histogram(raw[~raw.polarity.str.contains("O")], x="polarity", color="polarity")
fig.show()

fig = px.histogram(raw[~raw.category.str.contains("O")], x="category", color="category")
fig.show()

#Maps and Exctraction

In [9]:
POLARITY_MAP = {"Neg": 0, "Pos": 1}

In [10]:
def extract_aspect(token_df):
    examples = []
    dropped = []

    for sentence_id, group in token_df.groupby("sentence_id", sort=False):
        group = group.sort_values("source_row")
        sentence = " ".join(group["word"])
        active = None

        def finish_aspect():
            nonlocal active
            if active is not None:
                if active["valid"]:
                    examples.append({
                        "sentence_id": sentence_id,
                        "sentence": sentence,
                        "aspect": " ".join(active["words"]),
                        "category": active["category"],
                        "polarity": active["polarity"],
                        "label": active["label"],
                        "aspect_token_count": len(active["words"]),
                    })
                else:
                    dropped.append({
                        "sentence_id": sentence_id,
                        "aspect": " ".join(active["words"]),
                        "category": active["category"],
                        "polarity": active["polarity"],
                        "reason": active["drop_reason"],
                    })
            active = None

        for row in group.itertuples(index=False):
            tag = row.bio_tag
            category = None if row.category == "O" else row.category
            polarity_label = POLARITY_MAP.get(row.polarity)

            if tag == "B":
                finish_aspect()
                is_valid = category is not None and polarity_label is not None
                active = {
                    "words": [row.word],
                    "category": category,
                    "polarity": row.polarity,
                    "label": polarity_label,
                    "valid": is_valid,
                    "drop_reason": None if is_valid else "missing category/polarity on B",
                }

            elif tag == "I":
                if active is None:
                    continue

                active["words"].append(row.word)

                if (category !=active["category"] or polarity_label !=active["label"]):
                    active["valid"] = False
                    active["drop_reason"] = (
                        "category/polarity mismatch inside BIO span"
                    )

            elif tag == "O":
                finish_aspect()

        finish_aspect()

    return pd.DataFrame(examples), pd.DataFrame(dropped)


In [11]:

examples_df,dropped_df = extract_aspect(raw)

# print("\nDropped items:", len(dropped_df))
# if not dropped_df.empty:
#     display(dropped_df["reason"].value_counts().to_frame("count").reset_index())
#     display(dropped_df.head(5))

print("Extracted aspect examples:", len(examples_df))
print("Unique sentences with valid aspect:", examples_df["sentence_id"].nunique())

examples_df.to_csv(OUTPUT_DIR / "extracted_examples.csv", index=False, encoding="utf-8-sig")

Extracted aspect examples: 9698
Unique sentences with valid aspect: 4351


# Sentence-ascpect input

In [12]:
examples_df["model_input"] = (
    examples_df["sentence"]
    + " [SEP] "
    + examples_df["aspect"]
)

In [13]:

print("Model input examples:")
display(examples_df[["sentence", "aspect", "category", "polarity", "model_input"]].head(3))

Model input examples:


,sentence,aspect,category,polarity,model_input
0,عمال شركات التوصيل أمثال ذبحونا ابعث لي لوكيشن...,ذبحونا,Service,Neg,عمال شركات التوصيل أمثال ذبحونا ابعث لي لوكيشن...
1,عمال شركات التوصيل أمثال ذبحونا ابعث لي لوكيشن...,لوكيشن,Location,Neg,عمال شركات التوصيل أمثال ذبحونا ابعث لي لوكيشن...
2,عمال شركات التوصيل أمثال ذبحونا ابعث لي لوكيشن...,وعناوينا,Location,Neg,عمال شركات التوصيل أمثال ذبحونا ابعث لي لوكيشن...


#SPLIT

In [14]:
def make_split(df):
    y = df["label"].to_numpy(dtype=np.int32)
    groups = df["sentence"].to_numpy()
    dummy_x = np.zeros((len(df), 1), dtype=np.int8)

    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    trainval_idx, test_idx = next(outer.split(dummy_x, y, groups))

    inner = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=SEED + 1)
    train_local, val_local = next(
        inner.split(
            np.zeros((len(trainval_idx), 1), dtype=np.int8),
            y[trainval_idx],
            groups[trainval_idx]
        )
    )

    train_idx = trainval_idx[train_local]
    val_idx = trainval_idx[val_local]

    #verify no sentence is shared between partitions
    train_groups = set(groups[train_idx])
    val_groups = set(groups[val_idx])
    test_groups = set(groups[test_idx])
    assert train_groups.isdisjoint(val_groups)
    assert train_groups.isdisjoint(test_groups)
    assert val_groups.isdisjoint(test_groups)

    return train_idx, val_idx, test_idx

In [15]:
train_idx, val_idx, test_idx = make_split(examples_df)

train_df = examples_df.iloc[train_idx]
val_df = examples_df.iloc[val_idx]
test_df = examples_df.iloc[test_idx]

In [16]:
X_train = train_df["model_input"].to_numpy()
X_val = val_df["model_input"].to_numpy()
X_test = test_df["model_input"].to_numpy()

y_train = train_df["label"].to_numpy(dtype=np.int32)
y_val = val_df["label"].to_numpy(dtype=np.int32)
y_test = test_df["label"].to_numpy(dtype=np.int32)

# examples_df["split"] = "train"
# examples_df.loc[val_idx, "split"] = "val"
# examples_df.loc[test_idx, "split"] = "test"
# examples_df.to_csv(OUTPUT_DIR / "split_dataset.csv", index=False, encoding="utf-8-sig")

In [17]:
split_summary = []
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    split_summary.append({
        "split": name,
        "aspect examples": len(part),
        "unique sentences": part["sentence_id"].nunique(),
        "percentage of examples": round(100 * len(part) / len(examples_df), 2),
        "negative": int((part["label"] == 0).sum()),
        "positive": int((part["label"] == 1).sum())
    })

pd.DataFrame(split_summary)

,split,aspect examples,unique sentences,percentage of examples,negative,positive
0,train,6749,3040,69.59,5080,1669
1,validation,998,448,10.29,743,255
2,test,1951,863,20.12,1451,500


#vector

In [18]:
MAX_LENGTH = 80

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=20_000,
    output_mode="int",
    output_sequence_length=MAX_LENGTH,
    standardize=None,
    split="whitespace",
    name="text_vectorization",
)

vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(256))

train_lengths = np.array([len(text.split()) for text in X_train])
print("Vocabulary size:", len(vectorizer.get_vocabulary()))
print("Sequence length:", MAX_LENGTH)
print("Training inputs longer than the sequence limit:", f"{100 * np.mean(train_lengths > MAX_LENGTH):.2f}%")
print("Training input length percentiles:", np.percentile(train_lengths, [50, 90, 95, 99]).round(1))

Vocabulary size: 13715
Sequence length: 80
Training inputs longer than the sequence limit: 0.00%
Training input length percentiles: [28. 49. 52. 56.]


#LSTM

In [19]:

model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string, name="sentence_aspect_input"),
        vectorizer,
        tf.keras.layers.Embedding(
            input_dim=len(vectorizer.get_vocabulary()),
            output_dim=128,
            mask_zero=True,
            name="word_embedding",
        ),
        tf.keras.layers.LSTM(64, name="lstm_encoder"),
        tf.keras.layers.Dropout(0.2, name="dropout"),
        tf.keras.layers.Dense(2, activation="softmax", name="polarity_output"),
    ],
    name="simpler_baseline_LSTM",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Model: "simpler_baseline_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 80)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ word_embedding (Embedding)      │ (None, 80, 128)        │     1,755,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_encoder (LSTM)             │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ polarity_output (Dense)         │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,805,058 (6.89 MB)

 Trainable params: 1,805,058 (6.89 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
class Validation(tf.keras.callbacks.Callback):
    def __init__(self, x_val, y_val):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.scores = []

    def on_epoch_end(self, epoch, logs=None):
        probabilities = self.model.predict(self.x_val, batch_size=256, verbose=0)
        predictions = np.argmax(probabilities, axis=1)
        score = precision_recall_fscore_support(
            self.y_val,
            predictions,
            labels=[0, 1],
            average="macro",
            zero_division=0,
        )[2]
        self.scores.append(float(score))
        if logs is not None:
            logs["val_macro_f1"] = float(score)
        print(f" - val_macro_f1: {score:.4f}")


chkpt=tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUTPUT_DIR / "lstm_model.keras"),
        monitor="val_macro_f1",
        mode="max",
        save_best_only=True,
        verbose=1,
    )

early=tf.keras.callbacks.EarlyStopping(
        monitor="val_macro_f1",
        mode="max",
        patience=2,
        restore_best_weights=True,
        verbose=1,
    )

callbacks = [
    Validation(X_val, y_val),
    chkpt,
    early,
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

pd.DataFrame(history.history).to_csv(OUTPUT_DIR / "history.csv", index_label="epoch")

Epoch 1/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.7827 - loss: 0.4728 - val_macro_f1: 0.8128

Epoch 1: val_macro_f1 improved from None to 0.81280, saving model to /content/drive/MyDrive/Simple/lstm_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Simple/lstm_model.keras
211/211 ━━━━━━━━━━━━━━━━━━━━ 23s 92ms/step - accuracy: 0.8311 - loss: 0.3850 - val_accuracy: 0.8717 - val_loss: 0.3073 - val_macro_f1: 0.8128
Epoch 2/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.9203 - loss: 0.1978 - val_macro_f1: 0.8390

Epoch 2: val_macro_f1 improved from 0.81280 to 0.83898, saving model to /content/drive/MyDrive/Simple/lstm_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Simple/lstm_model.keras
211/211 ━━━━━━━━━━━━━━━━━━━━ 23s 105ms/step - accuracy: 0.9230 - loss: 0.1828 - val_accuracy: 0.8858 - val_loss: 0.3262 - val_macro_f1: 0.8390
Epoch 3/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.9584 - loss: 0.1123 - val_m

In [21]:
test_preds = np.argmax(model.predict(X_test, batch_size=256, verbose=0), axis=1)
print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(classification_report(y_test, test_preds, target_names=["Negative", "Positive"], digits=4))

Test Accuracy: 0.8719
              precision    recall  f1-score   support

    Negative     0.8887    0.9462    0.9166      1451
    Positive     0.8079    0.6560    0.7241       500

    accuracy                         0.8719      1951
   macro avg     0.8483    0.8011    0.8203      1951
weighted avg     0.8680    0.8719    0.8672      1951



#To test model later on comments

In [22]:
import numpy as np
import tensorflow as tf

LABEL_NAMES = {0: "Negative", 1: "Positive"}

def load_sentiment_model(model_path="/content/drive/MyDrive/Simple/lstm_model.keras"):
    return tf.keras.models.load_model(model_path)

def predict_comment(model, comment, aspect=""):
    comment = comment.strip()
    aspect = aspect.strip()

    model_input = f"{comment} [SEP] {aspect}" if aspect else comment
    input_tensor = tf.constant([model_input], dtype=tf.string)

    probs = model.predict(input_tensor, verbose=0)[0]
    pred_idx = int(np.argmax(probs))

    return {
        "polarity": LABEL_NAMES[pred_idx],
        "confidence": float(probs[pred_idx]),
        "probabilities": {
            "Negative": float(probs[0]),
            "Positive": float(probs[1]),
        },
    }


In [23]:
loaded_model = load_sentiment_model("/content/drive/MyDrive/Simple/lstm_model.keras")

test_cases = [
    #Negative
    ("أسو خدمة بالابلكيشن الطلب تأخر ساعتين والمندوب ما يعرف العنوان", "أسو خدمة"),
    ("خدمتكم زفت والابلكيشن دايما يكنسل الطلب بدون سبب", "خدمتكم زفت"),

    #Positive
    ("والله احسن ابلكيشن توصيل الطلب وصل بسرعة والمندوب محترم وايد", "احسن ابلكيشن"),
    ("والله احسن ابلكيشن توصيل الطلب وصل بسرعة والمندوب محترم وايد", "محترم"),

    #Mixed
    ("الابلكيشن احسن وايد من غيره بس التوصيل تأخر ساعتين", "احسن وايد"),
    ("الابلكيشن احسن وايد من غيره بس التوصيل تأخر ساعتين", "ساعتين"),

]

for comment, aspect in test_cases:
    res = predict_comment(loaded_model, comment, aspect)
    print(f"Comment : {comment}")
    print(f"Aspect  : {aspect}")
    print(f"Result  : {res['polarity']} ({res['confidence']:.2%}) | "
          f"Neg: {res['probabilities']['Negative']:.3f}, Pos: {res['probabilities']['Positive']:.3f}\n")

Comment : أسو خدمة بالابلكيشن الطلب تأخر ساعتين والمندوب ما يعرف العنوان
Aspect  : أسو خدمة
Result  : Negative (99.84%) | Neg: 0.998, Pos: 0.002

Comment : خدمتكم زفت والابلكيشن دايما يكنسل الطلب بدون سبب
Aspect  : خدمتكم زفت
Result  : Negative (99.97%) | Neg: 1.000, Pos: 0.000

Comment : والله احسن ابلكيشن توصيل الطلب وصل بسرعة والمندوب محترم وايد
Aspect  : احسن ابلكيشن
Result  : Negative (96.15%) | Neg: 0.961, Pos: 0.039

Comment : والله احسن ابلكيشن توصيل الطلب وصل بسرعة والمندوب محترم وايد
Aspect  : محترم
Result  : Positive (75.17%) | Neg: 0.248, Pos: 0.752

Comment : الابلكيشن احسن وايد من غيره بس التوصيل تأخر ساعتين
Aspect  : احسن وايد
Result  : Positive (83.87%) | Neg: 0.161, Pos: 0.839

Comment : الابلكيشن احسن وايد من غيره بس التوصيل تأخر ساعتين
Aspect  : ساعتين
Result  : Negative (98.01%) | Neg: 0.980, Pos: 0.020

